# What one Grover oracle costs

Two oracles for the same nonogram, measured on their own rather than inside a Grover
circuit. It is self-contained: it needs `qiskit`, `qiskit-algorithms` and
`qiskit-ibm-runtime`, and nothing from the project repo.

**The two oracles.** Qiskit's `PhaseOracleGate` reduces the clue formula over its whole
`2^n` truth table and emits one multi-controlled Z per solution. Its gate count tracks the
number of solutions, it never reads a clue, and building it returns the solution set as a
side effect. The clue oracle here carries one flag qubit per line, raises it with one
multi-controlled X per pattern that line's clue allows, flips the phase on all flags at
once, then lowers the flags again. It is built from the clues in milliseconds and never
consults a solution.

**What is measured.** One oracle, transpiled alone. A Grover circuit repeats its oracle
once per round and puts the same diffuser beside it either way, so a whole-circuit count is
the oracle count times the round count plus an equal number of diffusers, and at 4x4 that
round count is 201. Comparing the circuits compares mostly diffusers. Comparing the oracles
asks the question, and costs a factor of the round count less to run.

**What is recorded.** Every transpiler seed, not the winner alone. The oracle's build time
apart from its transpile time. The logical width of each oracle rather than the padded
device width. The pattern count and summed control width each board's clues imply, which is
what the clue oracle's cost is a function of.

Three measurements follow: how the ratio behaves as the board grows, how it varies board to
board at one size, and how the build time of each oracle grows.

## 1. Environment

In [ ]:
# pip install qiskit qiskit-algorithms qiskit-ibm-runtime
import os

# Qiskit's own threading competes with the process pool below for the same cores, and one
# circuit per worker is the coarser and faster split.
os.environ.setdefault("QISKIT_NUM_PROCS", "1")
os.environ.setdefault("RAYON_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import qiskit, qiskit_algorithms, qiskit_ibm_runtime
from qiskit_ibm_runtime.fake_provider import FakeTorino

print("qiskit", qiskit.__version__)
print("qiskit-algorithms", qiskit_algorithms.__version__)
print("qiskit-ibm-runtime", qiskit_ibm_runtime.__version__)
print("cores", os.cpu_count())
print("target", FakeTorino().name, FakeTorino().num_qubits, "qubits")

## 2. Configuration

`whole_circuit` is the expensive arm and is off. Leave it off unless you want the
whole-circuit numbers for comparison: at 4x4 it multiplies the work by the round count.

`workers` defaults to a quarter of the cores, to 16 at most. This runs on a shared machine
and each worker drops its own scheduling priority, so the sweep yields to anything else
rather than competing with it. Raise it if the box is yours.

`per_solution_count` is the one that matters for the spread question. Sampling boards in
the order they are generated gives 82% single-solution boards, and the single-solution case
is the one whose answer is already known. Equal counts per solution class answer a question
the uniform sample cannot.

In [ ]:
CONFIG = {
    "seeds": 8,                 # transpiler seeds per oracle; every one is recorded
    "spare": 2,                 # idle qubits lent to synthesis for the big gates
    "device": True,             # False measures synthesis alone, with no coupling map
    "whole_circuit": False,     # True measures Grover rather than the oracle: far slower
    "workers": max(1, min((os.cpu_count() or 4) // 4, 16)),
    "timeout": 3600,            # seconds per board, 0 for none
    "skip_marking_above": 20,   # cells past which Qiskit's oracle is not measured
    "sample_seed": 20260930,
}

# Question 1: one board per (shape, solution count), smallest shapes first.
SCALING = {"min_cells": 4, "max_cells": 24, "max_solutions": 4, "counts_per_shape": 3,
           "avoid_determined": True}

# Question 2: equal numbers of boards per solution count at one shape.
SPREAD = {"shape": (4, 4), "sample": 4000, "per_solution_count": 30, "max_solutions": 4}

# Question 3: build time alone, with no transpilation, as far as patience allows.
BUILD = {"min_cells": 4, "max_cells": 26, "repeats": 1}

OUT_SCALING = "oracle-scaling.jsonl"
OUT_SPREAD = "oracle-spread.jsonl"
OUT_BUILD = "oracle-build.jsonl"
CONFIG

## 3. The measurement code, as a module the workers can import

Windows and macOS start a worker with `spawn` rather than `fork`, and a spawned process can
import a module but cannot reach functions defined in a notebook cell. So the compute lives
in `oracle_cost_worker.py`, written beside this notebook by the cell below and imported
back. Re-run this cell after editing it.

In [ ]:
MODULE_SRC = r"""
'''Everything a worker process needs, in a module it can import.

Written beside the notebook rather than defined in it: on Windows and macOS a new process
is started with `spawn` rather than `fork`, and a spawned child can import a module but
cannot reach functions defined in a notebook cell.

The quantity measured here is one oracle, not a whole Grover circuit. A Grover circuit
repeats its oracle once per round, so its gate count is the oracle count times the round
count plus the same number of diffusers, and the diffuser is the same multi-controlled Z
whichever oracle it sits beside. Dividing it out afterwards is arithmetic on two unknowns.
Transpiling the oracle alone answers the question being asked and costs a round count less.
'''
from __future__ import annotations

import itertools
import math
import os
import random
import signal
import time

os.environ.setdefault("QISKIT_NUM_PROCS", "1")
os.environ.setdefault("RAYON_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

#: Unix can cap a board with an alarm; Windows cannot, and says so rather than pretending.
CAN_TIME_OUT = hasattr(signal, "SIGALRM")

BASIS = ["cz", "rz", "sx", "x", "id"]
TWO_QUBIT_NAMES = ("ecr", "cz", "cx")


def _step_aside():
    '''Run behind whatever else the machine is doing.'''
    try:
        if hasattr(os, "nice"):
            os.nice(10)
        elif os.name == "nt":
            import ctypes

            below_normal = 0x00004000
            handle = ctypes.windll.kernel32.GetCurrentProcess()
            ctypes.windll.kernel32.SetPriorityClass(handle, below_normal)
    except Exception:
        pass  # a scheduler that refuses is not a reason to stop measuring


_step_aside()




def line_patterns(length, clue):
    '''Every filling of one line that matches its clue, as ints with bit 0 leftmost.'''
    if clue in ((0,), ()):
        return [0]
    out = []

    def place(block, start, bits):
        if block == len(clue):
            out.append(bits)
            return
        run = clue[block]
        rest = clue[block + 1:]
        room = sum(rest) + len(rest)
        for first in range(start, length - room - run + 1):
            mask = ((1 << run) - 1) << first
            place(block + 1, first + run + 1, bits | mask)

    place(0, 0, 0)
    return out


def line_clue(cells):
    '''The clue a filled line produces.'''
    out = []
    run = 0
    for cell in cells:
        if cell:
            run += 1
        elif run:
            out.append(run)
            run = 0
    if run:
        out.append(run)
    return tuple(out) if out else (0,)


def clues_of(bits, rows, cols):
    '''The row and column clues a filled grid produces.'''
    grid = [bits[r * cols:(r + 1) * cols] for r in range(rows)]
    return ([line_clue(row) for row in grid],
            [line_clue([grid[r][c] for r in range(rows)]) for c in range(cols)])


def solve(row_clues, col_clues, limit=64):
    '''Every grid the clues accept, by placing a row at a time and pruning the columns.'''
    rows, cols = len(row_clues), len(col_clues)
    row_options = [line_patterns(cols, c) for c in row_clues]
    col_options = [line_patterns(rows, c) for c in col_clues]
    if any(not o for o in row_options) or any(not o for o in col_options):
        return []

    found = []

    def walk(r, placed, col_live):
        if len(found) >= limit:
            return
        if r == rows:
            found.append(list(placed))
            return
        for pattern in row_options[r]:
            nxt = []
            for c in range(cols):
                bit = (pattern >> c) & 1
                live = [p for p in col_live[c] if ((p >> r) & 1) == bit]
                if not live:
                    break
                nxt.append(live)
            else:
                placed.append(pattern)
                walk(r + 1, placed, nxt)
                placed.pop()

    walk(0, [], col_options)
    return found


def boards(rows, cols, sample=None, seed=0):
    '''Distinct solvable puzzles of this shape, with their solution counts.'''
    cells = rows * cols
    seen = set()

    def emit(bits):
        row_clues, col_clues = clues_of(bits, rows, cols)
        key = (tuple(row_clues), tuple(col_clues))
        if key in seen:
            return None
        seen.add(key)
        answers = solve(row_clues, col_clues)
        return (row_clues, col_clues, len(answers)) if answers else None

    if sample is None and cells <= 20:
        for bits in itertools.product([0, 1], repeat=cells):
            got = emit(list(bits))
            if got:
                yield got
        return

    rng = random.Random(seed)
    for _ in range(sample or 2000):
        got = emit([rng.randint(0, 1) for _ in range(cells)])
        if got:
            yield got


def lines_of(row_clues, col_clues):
    '''Every row and column, as the cell indices it covers and the clue it must match.'''
    rows, cols = len(row_clues), len(col_clues)
    out = [([r * cols + c for c in range(cols)], tuple(row_clues[r])) for r in range(rows)]
    out += [([r * cols + c for r in range(rows)], tuple(col_clues[c])) for c in range(cols)]
    return out


def pattern_stats(row_clues, col_clues):
    '''What the clue oracle is built from, as numbers to plot the cost against.

    `patterns` is one multi-controlled X each. `mcx_width` sums each gate's control count,
    which is what synthesis actually charges for. `determined` marks a puzzle whose every
    line allows exactly one filling: such a board is read off its clues without search, and
    its clue oracle is the cheapest one that shape can have.
    '''
    lines = lines_of(row_clues, col_clues)
    counts = [len(line_patterns(len(q), clue)) for q, clue in lines]
    widths = [len(q) for q, _ in lines]
    return {
        "lines": len(lines),
        "patterns": sum(counts),
        "mcx_width": sum(c * w for c, w in zip(counts, widths)),
        "longest_line": max(widths),
        "determined": sum(counts) == len(lines),
    }


def optimal_rounds(solutions, cells):
    '''Grover iteration count, as the backend and the page both compute it.'''
    if solutions <= 0:
        return 0
    return max(1, math.floor(math.pi / 4 * math.sqrt(2 ** cells / solutions)))




def clue_oracle(row_clues, col_clues):
    '''A phase oracle that tests the clues: one flag qubit per line.

    For each line, one multi-controlled X per pattern that line's clue allows raises its
    flag; the patterns are mutually exclusive, so toggling acts as an OR. A phase flip
    conditioned on every flag marks the grids satisfying all of them. Repeating the pattern
    gates lowers the flags again -- left raised they stay entangled with the cells and the
    interference Grover depends on falls apart.
    '''
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import MCXGate, ZGate

    cells = len(row_clues) * len(col_clues)
    lines = lines_of(row_clues, col_clues)
    qc = QuantumCircuit(cells + len(lines), name="clue oracle")

    def mark():
        for flag, (qubits, clue) in enumerate(lines):
            for pattern in line_patterns(len(qubits), clue):
                qc.append(MCXGate(len(qubits), ctrl_state=pattern), qubits + [cells + flag])

    mark()
    qc.append(ZGate().control(len(lines) - 1), [cells + i for i in range(len(lines))])
    mark()
    return qc


def clue_formula(row_clues, col_clues):
    '''The clues as a boolean expression, which is what Qiskit's oracle is built from.'''
    terms = []
    for qubits, clue in lines_of(row_clues, col_clues):
        options = []
        for pattern in line_patterns(len(qubits), clue):
            literals = [("v%d" % q if (pattern >> i) & 1 else "~v%d" % q)
                        for i, q in enumerate(qubits)]
            options.append("(" + " & ".join(literals) + ")")
        terms.append("(" + " | ".join(options) + ")")
    return " & ".join(terms)


def marking_oracle(row_clues, col_clues):
    '''Qiskit's oracle, which compiles to one marked grid per solution.

    `PhaseOracleGate` reduces the formula over its whole 2^n truth table, so constructing
    it enumerates the grid space and returns the solution set on the way. The time that
    takes is reported separately from transpilation, because it is the part that cannot be
    paid by a machine that does not already have the answer.
    '''
    from qiskit.circuit.library import PhaseOracleGate

    return PhaseOracleGate(clue_formula(row_clues, col_clues))


def as_circuit(gate, spare):
    '''One oracle on its own qubits, plus idle ones for synthesis to borrow.'''
    from qiskit import QuantumCircuit

    qc = QuantumCircuit(gate.num_qubits + spare)
    qc.append(gate, range(gate.num_qubits))
    return qc


def grover_circuit(oracle, cells, rounds, spare):
    '''Grover over one oracle, for the whole-circuit arm.'''
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import ZGate

    qc = QuantumCircuit(oracle.num_qubits + spare, cells)
    qc.h(range(cells))
    for _ in range(rounds):
        qc.append(oracle, range(oracle.num_qubits))
        qc.h(range(cells))
        qc.x(range(cells))
        qc.append(ZGate().control(cells - 1), list(range(cells)))
        qc.x(range(cells))
        qc.h(range(cells))
    qc.measure(range(cells), range(cells))
    return qc




class Timeout(Exception):
    pass


def _alarm(signum, frame):
    raise Timeout()


def transpile_runs(circuit, device, seeds):
    '''Every seed, kept. Selecting one of these is a reporting choice, made later.

    Keeping all of them is what makes the selection auditable: best-of-N on a quantity that
    varies seed to seed is a biased estimate of the typical cost, and the bias cannot be
    judged from the winner alone.
    '''
    from qiskit import transpile

    backend = None
    if device:
        from qiskit_ibm_runtime.fake_provider import FakeTorino
        backend = FakeTorino()

    runs = []
    started = time.time()
    for seed in range(seeds):
        if backend is not None:
            flat = transpile(circuit, backend=backend, optimization_level=3,
                             seed_transpiler=seed)
        else:
            flat = transpile(circuit, basis_gates=BASIS, coupling_map=None,
                             optimization_level=3, seed_transpiler=seed)
        ops = flat.count_ops()
        runs.append({
            "seed": seed,
            "two_qubit": sum(v for g, v in ops.items() if g in TWO_QUBIT_NAMES),
            "depth": flat.depth(),
            "gates": sum(ops.values()),
        })
    return {"runs": runs, "seconds": round(time.time() - started, 2)}


def summarise(block, logical_qubits, build_seconds):
    '''One oracle's numbers: the shallowest run, and the range every run covers.'''
    runs = block["runs"]
    best = min(runs, key=lambda r: r["depth"])
    out = {
        "logical_qubits": logical_qubits,
        "build_seconds": round(build_seconds, 3),
        "transpile_seconds": block["seconds"],
        "seeds": len(runs),
        "runs": runs,
    }
    for field in ("two_qubit", "depth", "gates"):
        values = [r[field] for r in runs]
        out[field] = best[field]
        out[field + "_range"] = [min(values), max(values)]
    return out


def measure_board(job):
    '''One board, both oracles. Returns a plain dict, safe to append as JSON.'''
    key, row_clues, col_clues, solutions, cfg = job
    row_clues = [tuple(c) for c in row_clues]
    col_clues = [tuple(c) for c in col_clues]
    rows, cols = len(row_clues), len(col_clues)
    cells = rows * cols
    out = {
        "key": key, "rows": rows, "cols": cols, "cells": cells, "solutions": solutions,
        "rounds": optimal_rounds(solutions, cells),
        "device": cfg["device"], "seeds": cfg["seeds"], "spare": cfg["spare"],
        "scope": "whole_circuit" if cfg["whole_circuit"] else "oracle",
        "row_clues": [list(c) for c in row_clues],
        "col_clues": [list(c) for c in col_clues],
    }
    out.update(pattern_stats(row_clues, col_clues))

    capped = bool(cfg["timeout"]) and CAN_TIME_OUT
    if capped:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(cfg["timeout"])
    try:
        started = time.time()
        clue = clue_oracle(row_clues, col_clues)
        clue_built = time.time() - started
        clue_logical = clue.num_qubits

        subject = clue
        if cfg["whole_circuit"]:
            subject = grover_circuit(clue.to_gate(label="clue"), cells, out["rounds"], 0)
        out["clue"] = summarise(
            transpile_runs(as_circuit_or_self(subject, cfg["spare"]), cfg["device"],
                           cfg["seeds"]),
            clue_logical, clue_built)

        if cells <= cfg["skip_marking_above"]:
            started = time.time()
            marking = marking_oracle(row_clues, col_clues)
            marking_built = time.time() - started
            marking_logical = marking.num_qubits

            subject = marking
            if cfg["whole_circuit"]:
                subject = grover_circuit(marking, cells, out["rounds"], 0)
            out["marking"] = summarise(
                transpile_runs(as_circuit_or_self(subject, cfg["spare"]), cfg["device"],
                               cfg["seeds"]),
                marking_logical, marking_built)
            for field in ("two_qubit", "depth"):
                out["ratio_" + field] = round(
                    out["clue"][field] / out["marking"][field], 3)
    except Timeout:
        out["timeout"] = cfg["timeout"]
    except Exception as exc:
        out["error"] = "%s: %s" % (type(exc).__name__, exc)
    finally:
        if capped:
            signal.alarm(0)
    return out


def as_circuit_or_self(subject, spare):
    '''Pad a gate onto spare qubits; a circuit that already has them passes through.'''
    from qiskit import QuantumCircuit

    if isinstance(subject, QuantumCircuit):
        if not spare:
            return subject
        wide = QuantumCircuit(subject.num_qubits + spare, subject.num_clbits)
        wide.compose(subject, qubits=range(subject.num_qubits),
                     clbits=range(subject.num_clbits), inplace=True)
        return wide
    return as_circuit(subject, spare)
"""

from pathlib import Path

WORKER = Path("oracle_cost_worker.py")
WORKER.write_text(MODULE_SRC, encoding="utf-8")

import importlib, sys
sys.path.insert(0, str(WORKER.resolve().parent))
import oracle_cost_worker as W
W = importlib.reload(W)

print("wrote", WORKER.resolve())
plus = ([(1,), (3,), (1,)], [(1,), (3,), (1,)])
print("3x3 plus solves to", len(W.solve(*plus)), "grid")
print("3x3 clue oracle:", W.clue_oracle(*plus).num_qubits, "qubits")
print("3x3 clue stats:", W.pattern_stats(*plus))
print("per-board timeout available on this platform:", W.CAN_TIME_OUT)

## 4. Check both oracles before spending machine time

An oracle is only worth measuring if it marks the right set. Each one is put in a Grover
circuit of the ideal length, and the solutions should carry the textbook probability
`sin((2k+1)theta)^2`. An oracle marking the wrong grids, or leaving a flag entangled with
the cells, misses this by far more than the tolerance. Both oracles are checked, so the
comparison is between two circuits that agree on what they mark.

In [ ]:
import math

from qiskit.quantum_info import Statevector


def amplified(oracle, row_clues, col_clues, rounds):
    cells = len(row_clues) * len(col_clues)
    qc = W.grover_circuit(oracle, cells, rounds, 0)
    qc = qc.remove_final_measurements(inplace=False)
    return Statevector(qc).probabilities(range(cells))


def check(row_clues, col_clues):
    cells = len(row_clues) * len(col_clues)
    answers = W.solve(row_clues, col_clues)
    rounds = W.optimal_rounds(len(answers), cells)
    indices = []
    for answer in answers:
        index = 0
        for r, pattern in enumerate(answer):
            for c in range(len(col_clues)):
                if (pattern >> c) & 1:
                    index |= 1 << (r * len(col_clues) + c)
        indices.append(index)

    theta = math.asin(math.sqrt(len(answers) / 2 ** cells))
    ideal = math.sin((2 * rounds + 1) * theta) ** 2
    worst = 0.0
    for name, oracle in (("clue", W.clue_oracle(row_clues, col_clues).to_gate(label="clue")),
                         ("marking", W.marking_oracle(row_clues, col_clues))):
        probs = amplified(oracle, row_clues, col_clues, rounds)
        hit = sum(probs[i] for i in indices)
        delta = abs(hit - ideal)
        worst = max(worst, delta)
        print("%2d cells M=%d k=%2d  %-7s measured %.6f  textbook %.6f  delta %.1e"
              % (cells, len(answers), rounds, name, hit, ideal, delta))
    return worst


worst = max(
    check([(1,), (1,)], [(1,), (1,)]),
    check([(1,), (1,)], [(1,), (0,), (1,)]),
    check([(1,), (3,), (1,)], [(1,), (3,), (1,)]),
)
assert worst < 1e-9, "an oracle does not amplify what the clues accept"
print("\nboth oracles verified")

## 5. Measurement

Results are appended as JSON Lines and flushed per board, and a rerun skips keys already in
the file, so this can be interrupted, resumed, or extended without recomputing. A board
that fails to compile records the error and the sweep carries on.

The pool is shut down without waiting. A worker wedged inside Qiskit's synthesis cannot be
interrupted from here, and every finished result is already on disk, so waiting for it only
hangs the kernel.

In [ ]:
import json, time
from concurrent.futures import ProcessPoolExecutor, as_completed


def done_keys(path):
    '''Keys already in the output, so a rerun extends rather than repeats.'''
    if not os.path.exists(path):
        return set()
    keys = set()
    with open(path) as handle:
        for line in handle:
            try:
                keys.add(json.loads(line)["key"])
            except Exception:
                pass
    return keys


def run(jobs, path, workers):
    '''Measure every job not already in the file, flushing each result as it lands.'''
    have = done_keys(path)
    todo = [j for j in jobs if j[0] not in have]
    print("%d jobs, %d already done, %d to measure, %d workers"
          % (len(jobs), len(jobs) - len(todo), len(todo), workers))
    if not todo:
        return
    if not W.CAN_TIME_OUT and CONFIG["timeout"]:
        print("no SIGALRM on this platform: the per-board timeout will not fire")

    started = time.time()
    workers = max(1, min(workers, 61, len(todo)))
    pool = ProcessPoolExecutor(max_workers=workers, max_tasks_per_child=1)
    try:
        futures = {pool.submit(W.measure_board, job): job[0] for job in todo}
        with open(path, "a") as handle:
            for n, future in enumerate(as_completed(futures), 1):
                record = future.result()
                handle.write(json.dumps(record) + "\n")
                handle.flush()
                note = record.get("error") or record.get("timeout") or ""
                clue = record.get("clue", {}).get("two_qubit", "-")
                marking = record.get("marking", {}).get("two_qubit", "-")
                print("%4d/%d %-22s clue %-9s marking %-9s %5.0fs %s"
                      % (n, len(todo), record["key"], format(clue, ",") if clue != "-" else "-",
                         format(marking, ",") if marking != "-" else "-",
                         time.time() - started, note))
    finally:
        # Nothing here waits on a wedged worker: the results are already on disk.
        pool.shutdown(wait=False, cancel_futures=True)
    print("done in %.0f s" % (time.time() - started))

## 6. Question 1 -- does the ratio hold as the board grows?

One board per (shape, solution count), smallest shapes first. Transposes are dropped.

Boards whose every line allows exactly one filling are skipped where another is available.
Such a puzzle is read straight off its clues with no search, and its clue oracle is one
gate per line, the cheapest that shape admits. Measuring only those understates the clue
oracle. `determined` is recorded either way, so the choice is visible in the output.

In [ ]:
def shapes_up_to(min_cells, max_cells):
    out = [(r, c) for r in range(2, max_cells + 1) for c in range(r, max_cells + 1)
           if min_cells <= r * c <= max_cells]
    return sorted(out, key=lambda rc: (rc[0] * rc[1], rc))


def board_source(rows, cols):
    # Every board of a small shape; a sample of a large one, where every board is 2^n.
    sample = None if rows * cols <= 16 else 3000
    return W.boards(rows, cols, sample, CONFIG["sample_seed"])


def scaling_jobs():
    for rows, cols in shapes_up_to(SCALING["min_cells"], SCALING["max_cells"]):
        chosen, backup = {}, {}
        for row_clues, col_clues, solutions in board_source(rows, cols):
            if solutions > SCALING["max_solutions"] or solutions in chosen:
                continue
            if SCALING["avoid_determined"] and W.pattern_stats(
                    row_clues, col_clues)["determined"]:
                backup.setdefault(solutions, (row_clues, col_clues))
                continue
            chosen[solutions] = (row_clues, col_clues)
            if len(chosen) >= SCALING["counts_per_shape"]:
                break
        picked = dict(backup)
        picked.update(chosen)
        for solutions in sorted(picked)[:SCALING["counts_per_shape"]]:
            row_clues, col_clues = picked[solutions]
            yield ("scaling:%dx%d:M%d" % (rows, cols, solutions),
                   row_clues, col_clues, solutions, CONFIG)


print("shapes:", shapes_up_to(SCALING["min_cells"], SCALING["max_cells"]))

In [ ]:
run(list(scaling_jobs()), OUT_SCALING, CONFIG['workers'])

## 7. Question 2 -- what does the spread look like at one size?

Equal numbers of boards per solution count, rather than the first N the generator produces.
A uniform sample of 4x4 boards is 82% single-solution, and for a single solution Qiskit's
oracle is one multi-controlled Z whatever the solution is, so its cost cannot vary and
measuring more of those settles nothing. The question is whether it varies once there is
more than one grid to mark, and that needs boards with more than one.

In [ ]:
def spread_jobs():
    rows, cols = SPREAD["shape"]
    wanted = SPREAD["per_solution_count"]
    taken = {}
    for row_clues, col_clues, solutions in W.boards(
            rows, cols, SPREAD["sample"], CONFIG["sample_seed"]):
        if solutions > SPREAD["max_solutions"]:
            continue
        group = taken.setdefault(solutions, [])
        if len(group) < wanted:
            group.append((row_clues, col_clues))
    for solutions in sorted(taken):
        for index, (row_clues, col_clues) in enumerate(taken[solutions]):
            yield ("spread:%dx%d:M%d:%d" % (rows, cols, solutions, index),
                   row_clues, col_clues, solutions, CONFIG)


from collections import Counter

jobs = list(spread_jobs())
print("boards per solution count:", dict(sorted(Counter(j[3] for j in jobs).items())))

In [ ]:
run(jobs, OUT_SPREAD, CONFIG['workers'])

## 8. Question 3 -- how does build time grow?

Construction alone, with no transpilation. `PhaseOracleGate` must materialise a `2^n` truth
table and reduce it, and it hands back the solution set on the way; the clue oracle is
assembled from the clues. This is the comparison that does not depend on a transpiler, a
seed, a coupling map or this machine's load, so it is the one that carries the argument.
The transpile timings elsewhere in this notebook illustrate it rather than establish it.

This cell runs in the kernel: it is a few seconds per board until the truth table gets
large, and it stops at the first board that exceeds the cap.

In [ ]:
import time

def build_timings(min_cells, max_cells, cap_seconds=900):
    out = []
    for rows, cols in shapes_up_to(min_cells, max_cells):
        board = None
        for row_clues, col_clues, solutions in W.boards(rows, cols, 400, CONFIG["sample_seed"]):
            if not W.pattern_stats(row_clues, col_clues)["determined"]:
                board = (row_clues, col_clues, solutions)
                break
        if board is None:
            continue
        row_clues, col_clues, solutions = board
        stats = W.pattern_stats(row_clues, col_clues)

        started = time.time()
        W.clue_oracle(row_clues, col_clues)
        clue_seconds = time.time() - started

        marking_seconds = None
        started = time.time()
        try:
            W.marking_oracle(row_clues, col_clues)
            marking_seconds = time.time() - started
        except Exception as exc:
            print("marking oracle failed at %dx%d: %s" % (rows, cols, exc))

        record = {"key": "build:%dx%d" % (rows, cols), "rows": rows, "cols": cols,
                  "cells": rows * cols, "solutions": solutions,
                  "clue_build_seconds": round(clue_seconds, 4),
                  "marking_build_seconds": (round(marking_seconds, 4)
                                            if marking_seconds else None)}
        record.update(stats)
        out.append(record)
        print("%2d cells  clue %8.4f s   marking %10s s   patterns %d"
              % (record["cells"], clue_seconds,
                 ("%.4f" % marking_seconds) if marking_seconds else "-", stats["patterns"]))
        with open(OUT_BUILD, "a") as handle:
            handle.write(json.dumps(record) + "\n")
        if marking_seconds and marking_seconds > cap_seconds:
            print("marking build passed the cap; stopping here")
            break
    return out


build = build_timings(BUILD["min_cells"], BUILD["max_cells"])

## 9. Read the results

In [ ]:
import statistics


def load(path):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path) as handle:
        for line in handle:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def summarize(path):
    rows = [r for r in load(path) if "clue" in r]
    failed = len(load(path)) - len(rows)
    print("%s: %d measured, %d failed or timed out" % (path, len(rows), failed))
    if not rows:
        return
    print("%7s %3s %4s %5s %9s %22s %22s %9s" %
          ("shape", "M", "n", "pats", "det", "clue 2q min/best/max",
           "marking 2q min/best/max", "ratio"))
    for r in sorted(rows, key=lambda r: (r["cells"], r["solutions"], r["key"])):
        def band(block):
            if not block:
                return "-"
            low, high = block["two_qubit_range"]
            return "%s / %s / %s" % (format(low, ","), format(block["two_qubit"], ","),
                                     format(high, ","))
        print("%7s %3d %4d %5d %9s %22s %22s %9s" % (
            "%dx%d" % (r["rows"], r["cols"]), r["solutions"], r["cells"], r["patterns"],
            "yes" if r["determined"] else "no",
            band(r.get("clue")), band(r.get("marking")),
            ("%.2fx" % r["ratio_two_qubit"]) if "ratio_two_qubit" in r else "-"))


for path in (OUT_SCALING, OUT_SPREAD):
    summarize(path)
    print()

## 10. Figures

Three charts, one claim each: what the clue oracle's cost is a function of, how the ratio
behaves across sizes, and how build time diverges. Each plots both oracles, because a chart
of one of them cannot show a difference between them.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed; skipping the figures")
    plt = None

CLUE, MARK = "#1a1a1a", "#c0392b"


def tidy(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="y", lw=0.4, color="#dddddd")
    ax.set_axisbelow(True)


if plt:
    spread = [r for r in load(OUT_SPREAD) if "clue" in r and "marking" in r]
    if spread:
        fig, ax = plt.subplots(figsize=(7.4, 4.2))
        for m in sorted({r["solutions"] for r in spread}):
            group = [r for r in spread if r["solutions"] == m]
            ax.plot([r["mcx_width"] for r in group],
                    [r["clue"]["two_qubit"] for r in group], "o", ms=5, alpha=0.8,
                    color=CLUE, mfc="none" if m > 1 else CLUE)
            ax.plot([r["mcx_width"] for r in group],
                    [r["marking"]["two_qubit"] for r in group], "s", ms=4, alpha=0.8,
                    color=MARK, mfc="none" if m > 1 else MARK)
        ax.set_xlabel("summed control width of the clue oracle's gates")
        ax.set_ylabel("two-qubit gates, one oracle")
        ax.set_title("%dx%d, %d boards: filled M=1, open M>1"
                     % (spread[0]["rows"], spread[0]["cols"], len(spread)))
        ax.legend(["clue oracle", "answer-marking oracle"], frameon=False)
        tidy(ax)
        plt.tight_layout()
        plt.show()

    scaling = [r for r in load(OUT_SCALING) if "ratio_two_qubit" in r]
    if scaling:
        fig, ax = plt.subplots(figsize=(7.4, 4.2))
        for m in sorted({r["solutions"] for r in scaling}):
            group = sorted([r for r in scaling if r["solutions"] == m],
                           key=lambda r: r["cells"])
            ax.plot([r["cells"] for r in group], [r["ratio_two_qubit"] for r in group],
                    "o-", ms=4, lw=0.8, label="M = %d" % m)
        ax.set_xlabel("cells on the board")
        ax.set_ylabel("clue two-qubit / marking two-qubit, one oracle")
        ax.set_ylim(bottom=0)
        ax.set_title("Cost ratio by board size and solution count")
        ax.legend(frameon=False)
        tidy(ax)
        plt.tight_layout()
        plt.show()

    built = load(OUT_BUILD)
    if built:
        fig, ax = plt.subplots(figsize=(7.4, 4.2))
        ax.plot([r["cells"] for r in built], [r["clue_build_seconds"] for r in built],
                "o-", ms=4, lw=0.8, color=CLUE, label="clue oracle")
        marking = [r for r in built if r["marking_build_seconds"]]
        ax.plot([r["cells"] for r in marking], [r["marking_build_seconds"] for r in marking],
                "o-", ms=4, lw=0.8, color=MARK, label="answer-marking oracle")
        ax.set_yscale("log")
        ax.set_xlabel("cells on the board")
        ax.set_ylabel("seconds to construct the oracle")
        ax.set_title("Construction time, with no transpiler involved")
        ax.legend(frameon=False)
        tidy(ax)
        plt.tight_layout()
        plt.show()

## 11. What to send back

All three `.jsonl` files. Every record carries its own clue pair, solution count, pattern
statistics, seeds and settings, so the results can be folded in without re-deriving
anything or guessing what produced them.

Worth noting when you report: whether `CONFIG["device"]` was on, whether `whole_circuit`
was on, what `seeds` was, and which shapes hit `skip_marking_above` so their ratio is
absent rather than estimated.